# EDA Malnutrición — PMCI Fundación Canguro

**Objetivo**: Análisis exploratorio orientado a los tres ejes del proyecto:
1. Cobertura de variables por periodo histórico (quinquenios)
2. Distribución de outcomes de malnutrición a 12 meses de edad corregida
3. Análisis de deserción y sesgo de selección
4. Mapa de features por fase temporal (nacimiento → 9 meses)
5. Correlaciones tempranas con malnutrición

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

PATH = "/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/KMC-70k-93-2024-Malnutricion-conVel-DATA-SPSS-20250322.xlsx"
OUT  = "/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/"

df_raw = pd.read_excel(PATH)
df = df_raw.replace('#NULL!', np.nan).copy()

# Convertir a numérico (pandas 2.x: errors='ignore' eliminado)
for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    # Solo reemplazar si al menos el 50% de los no-nulos se convierten con éxito
    n_orig = df[col].notna().sum()
    n_conv = converted.notna().sum()
    if n_orig == 0 or n_conv / n_orig >= 0.5:
        df[col] = converted

print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')
print(f'pandas: {pd.__version__} | numpy: {np.__version__}')

Dimensiones: 64,801 filas x 753 columnas
pandas: 3.0.1 | numpy: 2.4.2


## 0. Mapeo de periodos históricos

In [2]:
# Periodos históricos del PMCI
# P1-P3: registros históricos sin fecha de parto (probablemente 1998-2006)
# P4: 2007-2012 | P5: 2013-2017 | P6: 2018-2023
PERIOD_LABELS = {
    1.0: 'P1 (~1998-2001)',
    2.0: 'P2 (~2002-2004)',
    3.0: 'P3 (~2005-2006)',
    4.0: 'P4 (2007-2012)',
    5.0: 'P5 (2013-2017)',
    6.0: 'P6 (2018-2023)',
}
PERIOD_COLORS = ['#8e44ad','#2980b9','#27ae60','#f39c12','#e67e22','#e74c3c']

df['periodo_label'] = df['periodosanalisis'].map(PERIOD_LABELS)

conteo = df['periodosanalisis'].value_counts().sort_index()
print('Distribucion por periodo:')
for p, n in conteo.items():
    lbl = PERIOD_LABELS.get(p, str(p))
    print(f'  {lbl}: {n:,} pacientes ({n/len(df)*100:.1f}%)')
print(f'  Sin periodo asignado: {df["periodosanalisis"].isna().sum():,}')

# Grafica de distribucion por periodo
fig, ax = plt.subplots(figsize=(10, 4))
labels_p = [PERIOD_LABELS.get(p, str(p)) for p in conteo.index]
bars = ax.bar(labels_p, conteo.values,
              color=PERIOD_COLORS[:len(conteo)], edgecolor='white', width=0.6)
for bar, v in zip(bars, conteo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{v:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Distribucion de pacientes por periodo historico del PMCI',
             fontweight='bold')
ax.set_ylabel('Numero de pacientes')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xticklabels(labels_p, rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUT + 'mal_00_distribucion_periodos.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_00_distribucion_periodos.png')

Distribucion por periodo:
  P1 (~1998-2001): 4,406 pacientes (6.8%)
  P2 (~2002-2004): 3,919 pacientes (6.0%)
  P3 (~2005-2006): 4,315 pacientes (6.7%)
  P4 (2007-2012): 9,843 pacientes (15.2%)
  P5 (2013-2017): 18,965 pacientes (29.3%)
  P6 (2018-2023): 19,347 pacientes (29.9%)
  Sin periodo asignado: 4,006
Guardado: mal_00_distribucion_periodos.png


## 1. Variables Objetivo: Malnutrición a 12 meses

In [3]:
# Crear variables binarias de malnutricion a 12 meses
df['stunting12m']      = np.where(df['zscoretalla12cat'].notna(),
                                   (df['zscoretalla12cat'] == 1.0).astype(float), np.nan)
df['underweight12m_b'] = np.where(df['zscorepeso12cat'].notna(),
                                   (df['zscorepeso12cat'] == 1.0).astype(float), np.nan)
df['wasting12m']       = np.where(df['zscorepesotalla12cat'].notna(),
                                   (df['zscorepesotalla12cat'] == 1.0).astype(float), np.nan)

OUTCOMES_BIN = {
    'Stunting (HAZ<-2)':   ('stunting12m',      'zscoretalla12cat',    '#e74c3c'),
    'Bajo peso (WAZ<-2)':  ('underweight12m_b', 'zscorepeso12cat',     '#e67e22'),
    'Wasting (WHZ<-2)':    ('wasting12m',        'zscorepesotalla12cat','#f39c12'),
}

print('=' * 62)
print('OUTCOMES DE MALNUTRICION A 12 MESES CORREGIDOS')
print('=' * 62)
for nombre, (bin_col, cat_col, _) in OUTCOMES_BIN.items():
    total     = len(df)
    con_dato  = df[bin_col].notna().sum()
    nulos     = total - con_dato
    positivos = df[bin_col].sum()
    pct_pos   = positivos / con_dato * 100 if con_dato > 0 else 0
    ratio     = (con_dato - positivos) / positivos if positivos > 0 else float('inf')
    print(f'\n{nombre}')
    print(f'  Con dato  : {con_dato:>6,} ({con_dato/total*100:.1f}%)')
    print(f'  Nulos     : {nulos:>6,} ({nulos/total*100:.1f}%) <- desercion del programa')
    print(f'  Positivos : {positivos:>6,.0f} ({pct_pos:.1f}% de los con dato)')
    print(f'  Desbalance: {ratio:.1f}:1 (neg:pos)')

OUTCOMES DE MALNUTRICION A 12 MESES CORREGIDOS

Stunting (HAZ<-2)
  Con dato  : 30,953 (47.8%)
  Nulos     : 33,848 (52.2%) <- desercion del programa
  Positivos :  7,623 (24.6% de los con dato)
  Desbalance: 3.1:1 (neg:pos)

Bajo peso (WAZ<-2)
  Con dato  : 29,897 (46.1%)
  Nulos     : 34,904 (53.9%) <- desercion del programa
  Positivos :  3,239 (10.8% de los con dato)
  Desbalance: 8.2:1 (neg:pos)

Wasting (WHZ<-2)
  Con dato  : 29,828 (46.0%)
  Nulos     : 34,973 (54.0%) <- desercion del programa
  Positivos :  1,232 (4.1% de los con dato)
  Desbalance: 23.2:1 (neg:pos)


In [4]:
# Grafica: distribucion de outcomes categoricos
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cat_labels = {
    1.0: '< -2 SD\n(Malnutricion)',
    2.0: '-2 a +2 SD\n(Normal)',
    3.0: '> +2 SD\n(Sobrepeso)',
    4.0: '> +3 SD\n(Obesidad)'
}
colors_cat = ['#e74c3c','#2ecc71','#f39c12','#e67e22']

for ax, (titulo, (bin_col, cat_col, color)) in zip(axes, OUTCOMES_BIN.items()):
    data  = df[cat_col].dropna()
    vals  = data.value_counts().sort_index()
    lbls  = [cat_labels.get(k, str(k)) for k in vals.index]
    bars  = ax.bar(lbls, vals.values,
                   color=colors_cat[:len(vals)], edgecolor='white', width=0.6)
    for bar, v in zip(bars, vals.values):
        pct = v / len(data) * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    nulos = df[cat_col].isna().sum()
    ax.set_title(f'{titulo}\n(nulos: {nulos:,} = {nulos/len(df)*100:.0f}%)',
                 fontsize=10, fontweight='bold')
    ax.set_ylabel('Pacientes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Distribucion de Outcomes de Malnutricion a 12 meses EC',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT + 'mal_01_outcomes_distribucion.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_01_outcomes_distribucion.png')

Guardado: mal_01_outcomes_distribucion.png


## 2. Análisis de Deserción (Dropout)

In [5]:
# Embudo de retencion por visita
visit_map = {
    'Llego 40 sem':   'vino40',
    'Llego 3 meses':  'vino3m',
    'Llego 6 meses':  'vino6m',
    'Llego 9 meses':  'vino9m',
    'Llego 12 meses': 'vino12m',
}
retencion = {}
print('=== Retencion en el programa por visita ===')
for label, col in visit_map.items():
    if col in df.columns:
        n_vino = (df[col] == 1).sum()
        pct    = n_vino / len(df) * 100
        retencion[label] = pct
        print(f'  {label:<18}: {n_vino:>6,} vino ({pct:.1f}% del total)')

fig, ax = plt.subplots(figsize=(10, 4))
labels_r = list(retencion.keys())
vals_r   = list(retencion.values())
cols_r   = ['#2ecc71' if v >= 70 else '#f39c12' if v >= 50 else '#e74c3c' for v in vals_r]
bars = ax.barh(labels_r, vals_r, color=cols_r, edgecolor='white', height=0.5)
for bar, v in zip(bars, vals_r):
    ax.text(v + 0.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', fontweight='bold')
ax.set_xlim(0, 110)
ax.axvline(50, color='red', linestyle='--', alpha=0.5, label='50%')
ax.set_xlabel('% del total de pacientes')
ax.set_title('Embudo de Retencion en el PMCI', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUT + 'mal_02_retencion_embudo.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_02_retencion_embudo.png')

=== Retencion en el programa por visita ===
  Llego 40 sem      : 51,460 vino (79.4% del total)
  Llego 3 meses     : 45,174 vino (69.7% del total)
  Llego 6 meses     : 38,223 vino (59.0% del total)
  Llego 9 meses     : 32,503 vino (50.2% del total)
  Llego 12 meses    : 30,383 vino (46.9% del total)


Guardado: mal_02_retencion_embudo.png


In [6]:
# Comparar completadores vs desertores en variables de nacimiento
vars_comp = ['gestasal','ERN_Peso','ERN_Talla','ERN_Sexo','SGAprema',
             'menosde31sem','BPN','cesarea','CP_edadmaterna',
             'HD_DiasUCI','HD_TotalDiasHospital']
vars_ok = [v for v in vars_comp if v in df.columns]

df['completo_12m'] = (df['vino12m'] == 1).astype(int)
comp   = df[df['completo_12m'] == 1]
desert = df[df['completo_12m'] == 0]

print(f'Completo 12m : {len(comp):,} ({len(comp)/len(df)*100:.1f}%)')
print(f'Desertor     : {len(desert):,} ({len(desert)/len(df)*100:.1f}%)')
print()
print(f'{"Variable":<35} {"Completa 12m":>14} {"Desertor":>12} {"Diferencia":>12}')
print('-' * 75)
rows_graf = []
for v in vars_ok:
    mc = comp[v].mean()
    md = desert[v].mean()
    if not (np.isnan(mc) or np.isnan(md)):
        print(f'  {v:<33} {mc:>14.3f} {md:>12.3f} {mc-md:>+12.3f}')
        rows_graf.append((v, mc, md))

Completo 12m : 30,383 (46.9%)
Desertor     : 34,418 (53.1%)

Variable                              Completa 12m     Desertor   Diferencia
---------------------------------------------------------------------------
  gestasal                                  34.679       34.713       -0.034
  ERN_Peso                                2030.784     2052.345      -21.561
  ERN_Talla                                 44.221       44.420       -0.199
  ERN_Sexo                                   1.543        1.533       +0.010
  SGAprema                                   1.419        1.414       +0.005
  menosde31sem                               0.091        0.087       +0.005
  BPN                                        0.236        0.240       -0.004
  cesarea                                    0.631        0.630       +0.001
  CP_edadmaterna                            27.906       26.875       +1.031
  HD_DiasUCI                                 3.545        3.471       +0.073
  HD_TotalDiasHo

In [7]:
if rows_graf:
    nombres, vals_c, vals_d = zip(*rows_graf[:10])
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(nombres))
    w = 0.35
    ax.bar(x - w/2, vals_c, w, label='Completo 12m', color='#2ecc71', edgecolor='white')
    ax.bar(x + w/2, vals_d, w, label='Desertor',     color='#e74c3c', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(nombres, rotation=30, ha='right', fontsize=9)
    ax.set_title('Variables de nacimiento: Completadores vs Desertores',
                 fontweight='bold')
    ax.set_ylabel('Media')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT + 'mal_03_desercion_comparacion.png', bbox_inches='tight')
    plt.close()
    print('Guardado: mal_03_desercion_comparacion.png')

Guardado: mal_03_desercion_comparacion.png


In [8]:
# Retencion a 12m por periodo
periodos_validos = sorted(df['periodosanalisis'].dropna().unique())

ret_p = []
for p in periodos_validos:
    sub = df[df['periodosanalisis'] == p]
    n_vino = (sub['vino12m'] == 1).sum()
    n_base = sub['vino12m'].notna().sum()
    pct = n_vino / n_base * 100 if n_base > 0 else 0
    ret_p.append({'periodo': PERIOD_LABELS[p], 'pct': pct, 'color': PERIOD_COLORS[int(p)-1]})

fig, ax = plt.subplots(figsize=(10, 4))
lbls_p  = [r['periodo'] for r in ret_p]
pcts_p  = [r['pct']    for r in ret_p]
cols_p  = [r['color']  for r in ret_p]
bars = ax.bar(lbls_p, pcts_p, color=cols_p, edgecolor='white', width=0.6)
for bar, v in zip(bars, pcts_p):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.axhline(66.7, color='gray', linestyle='--', alpha=0.7, label='Promedio global (66.7%)')
ax.set_ylim(0, 110)
ax.set_ylabel('% que llego al control 12m')
ax.set_title('Retencion hasta 12 meses por periodo historico', fontweight='bold')
ax.set_xticklabels(lbls_p, rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(OUT + 'mal_04_retencion_por_periodo.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_04_retencion_por_periodo.png')

Guardado: mal_04_retencion_por_periodo.png


## 3. Análisis por Periodos — Cobertura de Variables

**¿Qué variables estaban disponibles en cada periodo histórico?**  
Los nulos altos en algunos features no son solo deserción — reflejan variables que se empezaron a medir en protocolos recientes o que fueron descontinuadas.

In [9]:
# Calcular cobertura (% no-nulo) por variable y por periodo
skip = {'Idenfinal','Iden_Codigo','Iden_Sede','Iden_FechaParto',
        'periodosanalisis','periodo_label','completo_12m',
        'stunting12m','underweight12m_b','wasting12m'}
# Excluir tambien variables de 12m (data leakage futuro)
skip |= {c for c in df.columns if '12' in str(c)}

feature_cols = [c for c in df.columns if c not in skip]

cobertura = {}
for p in periodos_validos:
    sub = df[df['periodosanalisis'] == p][feature_cols]
    cobertura[PERIOD_LABELS[p]] = (sub.notna().sum() / len(sub) * 100)

cov_df = pd.DataFrame(cobertura)
period_cols_only = [PERIOD_LABELS[p] for p in periodos_validos]

cov_df['min_cob'] = cov_df[period_cols_only].min(axis=1)
cov_df['max_cob'] = cov_df[period_cols_only].max(axis=1)
cov_df['rango']   = cov_df['max_cob'] - cov_df['min_cob']

n_univ    = (cov_df['min_cob'] >= 50).sum()
n_parcial = ((cov_df['min_cob'] < 50) & (cov_df['max_cob'] >= 50)).sum()
n_recient = ((cov_df['max_cob'] >= 50) & (cov_df['min_cob'] < 10)).sum()
n_sin     = (cov_df['max_cob'] < 10).sum()

print('=== Clasificacion de variables por disponibilidad historica ===')
print(f'  Universales (>=50% en TODOS los periodos) : {n_univ:>4} vars')
print(f'  Parciales   (>=50% en ALGUNOS periodos)   : {n_parcial:>4} vars')
print(f'  Solo recien (alta cob solo en P5-P6)      : {n_recient:>4} vars')
print(f'  Sin datos   (<10% en todos los periodos)  : {n_sin:>4} vars')
print(f'  Total features analizados: {len(feature_cols)}')

=== Clasificacion de variables por disponibilidad historica ===
  Universales (>=50% en TODOS los periodos) :  156 vars
  Parciales   (>=50% en ALGUNOS periodos)   :  457 vars
  Solo recien (alta cob solo en P5-P6)      :  403 vars
  Sin datos   (<10% en todos los periodos)  :   46 vars
  Total features analizados: 705


In [10]:
# Heatmap: top 60 variables con mayor variacion entre periodos
cambio = cov_df[cov_df['rango'] > 30].sort_values('rango', ascending=False)
top60  = cambio.head(60)[period_cols_only]

fig, ax = plt.subplots(figsize=(14, 18))
sns.heatmap(top60,
            cmap='RdYlGn', vmin=0, vmax=100,
            annot=True, fmt='.0f', annot_kws={'size': 7},
            linewidths=0.3, linecolor='gray',
            cbar_kws={'label': '% datos disponibles'},
            ax=ax)
ax.set_title(
    'Top 60 variables con mayor cambio entre periodos\n'
    'Verde = disponible | Rojo = no medida en ese periodo',
    fontweight='bold', fontsize=12)
ax.set_xlabel('Periodo historico')
ax.set_ylabel('Variable')
ax.tick_params(axis='y', labelsize=7)
ax.tick_params(axis='x', labelsize=9, rotation=20)
plt.tight_layout()
plt.savefig(OUT + 'mal_05_cobertura_heatmap_cambio.png', bbox_inches='tight')
plt.close()
print(f'Guardado: mal_05_cobertura_heatmap_cambio.png')
print(f'Variables con cambio de protocolo (rango > 30pp): {len(cambio)}')

Guardado: mal_05_cobertura_heatmap_cambio.png
Variables con cambio de protocolo (rango > 30pp): 467


In [11]:
# Heatmap: variables universales (candidatas prioritarias para el modelo)
vars_universales = cov_df[cov_df['min_cob'] >= 50].index.tolist()
top_univ = (cov_df.loc[vars_universales, period_cols_only]
              .assign(min=lambda d: d.min(axis=1))
              .sort_values('min', ascending=False)
              .drop(columns='min')
              .head(50))

fig, ax = plt.subplots(figsize=(14, 14))
sns.heatmap(top_univ,
            cmap='YlGn', vmin=50, vmax=100,
            annot=True, fmt='.0f', annot_kws={'size': 7},
            linewidths=0.3, linecolor='lightgray',
            cbar_kws={'label': '% datos disponibles (>=50%)'},
            ax=ax)
ax.set_title(
    f'Top 50 variables universales (n total: {len(vars_universales)})\n'
    'Disponibles >=50% en TODOS los periodos — candidatas prioritarias para el modelo',
    fontweight='bold', fontsize=12)
ax.tick_params(axis='y', labelsize=7)
ax.tick_params(axis='x', labelsize=9, rotation=20)
plt.tight_layout()
plt.savefig(OUT + 'mal_06_variables_universales.png', bbox_inches='tight')
plt.close()
print(f'Guardado: mal_06_variables_universales.png')
print(f'Total variables universales: {len(vars_universales)}')
print('Lista completa:')
for v in sorted(vars_universales):
    print(f'  {v}: min={cov_df.loc[v,"min_cob"]:.0f}%')

Guardado: mal_06_variables_universales.png
Total variables universales: 156
Lista completa:
  ANOCAT: min=100%
  BMI2: min=76%
  BMInacermas2DE: min=97%
  BPN: min=96%
  CONSULT08: min=100%
  CONSULT09: min=100%
  CONSULT10: min=100%
  CONSULT11: min=100%
  CP_SA_InfGineco: min=77%
  CP_SA_InfUrinaria: min=77%
  CP_SA_Sangrado: min=77%
  CP_TotalCPN: min=94%
  CP_edadmaterna: min=91%
  CSP_EscolaridadMadre: min=78%
  CSP_EscolaridadPadre: min=75%
  CSP_SituaPareja: min=79%
  DIASTOT08: min=51%
  DIASTOT09: min=54%
  DIASTOT10: min=50%
  EG1: min=100%
  EG40: min=100%
  EGEnt: min=100%
  ERN_Ballard: min=100%
  ERN_LubchencoFenton: min=97%
  ERN_PC: min=77%
  ERN_Peso: min=100%
  ERN_Sexo: min=100%
  ERN_Talla: min=97%
  ERN_Talla0: min=97%
  HD_DiasOxigeno: min=80%
  HD_DiasVenMecanica: min=80%
  HD_Infecciones: min=73%
  HD_TotalDiasHospital: min=100%
  INFECCIONOSOCOMIAL: min=72%
  Iden_embarazoMultiple: min=98%
  Indexnutricion40sem: min=76%
  LBWI: min=100%
  MUERTE1ANO: min=98%
  

## 4. Mapa de Features por Fase Temporal

In [12]:
# Clasificacion de variables en fases temporales
FASES = {
    'F0_Prenatal_Parto': [
        'CSP_SituaPareja','CSP_TipoVivienda','CSP_EscolaridadMadre',
        'CSP_SituacionLaboralMadre','CSP_EscolaridadPadre','CSP_SituacionLaboralPadre',
        'CSP_IngresoMensual','CSP_NutricionFam','CSP_EmbarazoDeseado',
        'CP_PesoMadre','CP_TallaMadre','CP_tallamadremetro','BMImadre','BMImadrecat',
        'CP_PesoPadre','CP_TallaPadre','CP_TotalCPN','CP_ARO','CP_MesInicCP',
        'CP_SA_Sangrado','CP_SA_InfUrinaria','CP_SA_Anemia','CP_SA_APP','CP_SA_Preclampsia',
        'CP_rhMadre','CP_MadreAlcohol','CP_MadreDrogas','CP_MadreFumo',
        'CP_edadmaterna','edadmatcat','primipara',
        'PA_TipoParto','PA_NumDosisCorticoides','PA_ComplicacionsPartoPreeclampsia',
        'PA_DiasHospiMadre','cesarea','toxemia','anemiamadre','trimestre',
        'Iden_embarazoMultiple','Sistemadeaseguramiento',
    ],
    'F1_Nacimiento': [
        'ERN_Peso','ERN_Talla','ERN_Talla0','ERN_Sexo','ERN_PC',
        'ERN_A_1min','ERN_A_5min','ERN_A_10min','ERN_Ballard',
        'gestasal','gestacat','gestasalsindecimales','menosde31sem','Nearterm',
        'RCIUpeso','RCIUtalla','RCIUpesoytallanacer','RCIUPC',
        'zscorepesotalla0','zscorepesotalla0cat','zscorepeso0','zscorepeso0cat',
        'zscoretalla0','zscoretalla0cat','zscorePC0','zscorePC0cat',
        'SGAprema','BPN','PESO1500G','pesocat','ERN_sepsis',
        'ERN_LubchencoFenton','apgarcat1','apgarcat5',
    ],
    'F2_Hospitalizacion': [
        'HD_DiasVenMecanica','HD_DiasVenNoInvasiva','HD_DiasOxigeno','HD_DiasCanulaNasa',
        'HD_DiasFototerapia','HD_DiasUCI','HD_DiasURN','HD_TotalDiasHospital',
        'HD_DosisSurfactante','HD_CicloAntibio','HD_NumTrasSanguineas',
        'HD_ValorMasAltoBilirubina','HD_UltiValorHematocrito',
        'HD_C_HemorragiaIntra','HD_C_Hipoglicemia','HD_C_Apnea','HD_C_Ictericia',
        'HD_C_DisplasiaBronco','HD_C_OxigenoDependencia','HD_C_Convulsiones',
        'HD_PesoSalida','HD_TipoAlimentacionS',
        'UCI','ALIMENTAPARENTERAL','INFECCIONOSOCOMIAL','surfactante',
        'tipoventilacion','corticodosis','corticoprenatalmenos34',
        'problemaneurologico','anoxia5mn','indicadorHICentrada',
    ],
    'F3_40semanas': [
        'EGEnt','gestaentrada','gestaentradacat','PesosalidaPC','edadsalidaPC','edadgestasalPC',
        'EG1','zscorepesotallaOMS1','zscorepesotalla1','zscorepesotalla1cat',
        'zscorepeso1','zscorepeso1cat','zscoretalla1','zscoretalla1cat',
        'zscorePC1','zscorePC1cat','BMI1','zscoreBMI1',
        'gananciapesonacerpesoentradaPMC','gananciatallanacertallaentradaPMC',
        'algoLM40sem','algoLA40','alisalida','ali40','LME40',
        'Indexnutricion40sem','rehosp40','oxigenoalaentrada','RCEUFentonentrada',
        'vino40',
    ],
    'F4_3meses': [
        'EG40','zscorepesotalla2','zscorepesotalla2cat','zscorepeso2','zscorepeso2cat',
        'zscoretalla2','zscoretalla2cat','zscorePC2','zscorePC2cat',
        'zscorepesoOMS2','zscorepesoOMS2cat','zscoretallaOMS2','zscoretallaOMS2cat',
        'BMI2','zscoreBMI2',
        'velocidadzscorepeso40_3m','velocidadzscore3m_40semOMS',
        'gananciapesoentradapeso40sem','Gananciatallaentradatalla40sem',
        'ali3m','algoLM3meses','algoLA3m','LME3m',
        'infanib3m','DIASTOT08','REHOSP08','vino3m',
    ],
    'F5_6meses': [
        'zscorepesotalla6','zscorepesotalla6cat','zscorepeso6','zscorepeso6cat',
        'zscoretalla6','zscoretalla6cat','zscorePC6','zscorePC6cat',
        'velocidad6_3mesesOMS',
        'ali6m','algoLM6meses','algoLA6m','LME6m',
        'infanib6m','rsm6m','CD6','DIASTOT09','REHOSP09','vino6m',
    ],
    'F6_9meses': [
        'zscorepesotalla9','zscorepesotalla9cat','zscorepeso9','zscorepeso9cat',
        'zscoretalla9','zscoretalla9cat','zscorePC9','zscorePC9cat',
        'velocidad9_6mesesOMS',
        'ali9m','algoLA9m',
        'infanib9m','DIASTOT10','REHOSP10','vino9m',
    ],
}

FASES_OK = {}
print('Variables encontradas por fase:')
for fase, cols in FASES.items():
    presentes = [c for c in cols if c in df.columns]
    FASES_OK[fase] = presentes
    print(f'  {fase}: {len(presentes)}/{len(cols)} variables')

Variables encontradas por fase:
  F0_Prenatal_Parto: 41/41 variables
  F1_Nacimiento: 34/34 variables
  F2_Hospitalizacion: 32/32 variables
  F3_40semanas: 30/30 variables
  F4_3meses: 27/27 variables
  F5_6meses: 19/19 variables
  F6_9meses: 15/15 variables


In [13]:
# Heatmap: cobertura media por fase y por periodo
resumen_fases = []
for fase, cols in FASES_OK.items():
    if not cols:
        continue
    for p in periodos_validos:
        sub     = df[df['periodosanalisis'] == p][cols]
        cob_med = (sub.notna().sum() / len(sub) * 100).mean()
        resumen_fases.append({
            'Fase':    fase,
            'Periodo': PERIOD_LABELS[p],
            'Cobertura_media': cob_med,
        })

rf_df = pd.DataFrame(resumen_fases)
pivot = rf_df.pivot(index='Fase', columns='Periodo', values='Cobertura_media')
# Reordenar columnas cronologicamente
pivot = pivot[[PERIOD_LABELS[p] for p in periodos_validos if PERIOD_LABELS[p] in pivot.columns]]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot,
            cmap='RdYlGn', vmin=0, vmax=100,
            annot=True, fmt='.0f', annot_kws={'size': 10},
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': '% cobertura media'},
            ax=ax)
ax.set_title(
    'Cobertura media (%) por FASE TEMPORAL y PERIODO HISTORICO\n'
    'Verde = datos disponibles | Rojo = variables sin medir en ese periodo',
    fontweight='bold', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, rotation=20)
plt.tight_layout()
plt.savefig(OUT + 'mal_07_cobertura_fase_periodo.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_07_cobertura_fase_periodo.png')

Guardado: mal_07_cobertura_fase_periodo.png


## 5. Evolución de la Malnutrición por Periodo Histórico

In [14]:
prev_rows = []
for p in periodos_validos:
    sub = df[df['periodosanalisis'] == p]
    for nombre, (bin_col, _, _) in OUTCOMES_BIN.items():
        n_tot = sub[bin_col].notna().sum()
        n_pos = sub[bin_col].sum()
        if n_tot > 30:  # al menos 30 casos con dato
            prev_rows.append({
                'Periodo': PERIOD_LABELS[p],
                'Outcome': nombre,
                'Prevalencia': n_pos / n_tot * 100,
                'N': n_tot,
            })

prev_df   = pd.DataFrame(prev_rows)
pivot_pv  = prev_df.pivot(index='Periodo', columns='Outcome', values='Prevalencia')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Lineas de tendencia
ax = axes[0]
cols_out = ['#e74c3c','#e67e22','#f39c12']
for col_out, color in zip(pivot_pv.columns, cols_out):
    vals_ok = pivot_pv[col_out].dropna()
    ax.plot(vals_ok.index, vals_ok.values, marker='o', linewidth=2,
            color=color, label=col_out)
    for xi, yi in zip(vals_ok.index, vals_ok.values):
        ax.text(xi, yi + 0.4, f'{yi:.1f}%', ha='center', fontsize=8, color=color)
ax.set_title('Prevalencia de malnutricion por periodo', fontweight='bold')
ax.set_ylabel('Prevalencia (%)')
ax.set_xticklabels(pivot_pv.index, rotation=20, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Heatmap
ax2 = axes[1]
sns.heatmap(pivot_pv.T,
            cmap='YlOrRd', annot=True, fmt='.1f',
            cbar_kws={'label': 'Prevalencia (%)'},
            linewidths=0.5, ax=ax2)
ax2.set_title('Heatmap prevalencia x periodo', fontweight='bold')
ax2.tick_params(axis='x', labelsize=8, rotation=20)

plt.suptitle('Evolucion de la malnutricion a traves del tiempo (PMCI)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mal_08_prevalencia_por_periodo.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_08_prevalencia_por_periodo.png')
print()
print(pivot_pv.round(1).to_string())

Guardado: mal_08_prevalencia_por_periodo.png

Outcome          Bajo peso (WAZ<-2)  Stunting (HAZ<-2)  Wasting (WHZ<-2)
Periodo                                                                 
P1 (~1998-2001)                15.3               37.0               4.6
P2 (~2002-2004)                12.9               30.4               4.7
P3 (~2005-2006)                12.6               34.0               3.2
P4 (2007-2012)                 11.0               27.1               4.0
P5 (2013-2017)                 10.5               21.4               4.4
P6 (2018-2023)                  9.4               19.5               3.9


## 6. Correlaciones Tempranas con Malnutrición a 12m

In [15]:
# Variables tempranas (F0 + F1) vs outcomes de malnutricion
early_cols = FASES_OK['F0_Prenatal_Parto'] + FASES_OK['F1_Nacimiento']
early_cols = [c for c in early_cols if c in df.columns]

df_corr = df[early_cols + ['stunting12m','underweight12m_b','wasting12m']].copy()
for c in df_corr.columns:
    df_corr[c] = pd.to_numeric(df_corr[c], errors='coerce')

corr_s = df_corr[early_cols].corrwith(df_corr['stunting12m']).dropna()
corr_u = df_corr[early_cols].corrwith(df_corr['underweight12m_b']).dropna()
corr_w = df_corr[early_cols].corrwith(df_corr['wasting12m']).dropna()

top15 = corr_s.abs().sort_values(ascending=False).head(15)
print('Top 15 variables de nacimiento mas correlacionadas con STUNTING a 12m:')
for v in top15.index:
    print(f'  {v:<40}  r_stunting={corr_s[v]:+.3f}  r_undwt={corr_u.get(v,float("nan")):+.3f}  r_wasting={corr_w.get(v,float("nan")):+.3f}')

Top 15 variables de nacimiento mas correlacionadas con STUNTING a 12m:
  zscoretalla0                              r_stunting=-0.270  r_undwt=-0.199  r_wasting=-0.078
  zscorepeso0                               r_stunting=-0.242  r_undwt=-0.200  r_wasting=-0.100
  RCIUtalla                                 r_stunting=+0.237  r_undwt=+0.178  r_wasting=+0.069
  RCIUpesoytallanacer                       r_stunting=+0.234  r_undwt=+0.183  r_wasting=+0.074
  RCIUpeso                                  r_stunting=+0.205  r_undwt=+0.167  r_wasting=+0.079
  SGAprema                                  r_stunting=+0.198  r_undwt=+0.161  r_wasting=+0.074
  ERN_Talla                                 r_stunting=-0.193  r_undwt=-0.165  r_wasting=-0.089
  ERN_Talla0                                r_stunting=-0.193  r_undwt=-0.165  r_wasting=-0.089
  zscoretalla0cat                           r_stunting=-0.189  r_undwt=-0.152  r_wasting=-0.064
  ERN_Peso                                  r_stunting=-0.185  r_

In [16]:
top_vars = corr_s.abs().sort_values(ascending=False).head(15).index
corr_comp = pd.DataFrame({
    'Stunting':   corr_s[top_vars],
    'Bajo peso':  corr_u.reindex(top_vars),
    'Wasting':    corr_w.reindex(top_vars),
})

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(top_vars))
w = 0.25
for i, (col_n, color) in enumerate(zip(corr_comp.columns, ['#e74c3c','#e67e22','#f39c12'])):
    ax.bar(x + i*w, corr_comp[col_n].values, w,
           label=col_n, color=color, edgecolor='white')
ax.set_xticks(x + w)
ax.set_xticklabels(top_vars, rotation=35, ha='right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(
    'Correlacion de Pearson: variables de nacimiento vs malnutricion a 12m\n'
    '(top 15 por correlacion con stunting)',
    fontweight='bold')
ax.set_ylabel('Correlacion de Pearson')
ax.legend()
plt.tight_layout()
plt.savefig(OUT + 'mal_09_correlaciones_nacimiento_12m.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_09_correlaciones_nacimiento_12m.png')

Guardado: mal_09_correlaciones_nacimiento_12m.png


In [17]:
# Senal predictiva por fase: que fase aporta mas informacion sobre stunting?
# Usar df completo (con todas las variables de todas las fases)
fases_senal = []
for fase, cols in FASES_OK.items():
    if not cols:
        continue
    cols_num = [c for c in cols if c in df.columns]
    if not cols_num:
        continue
    # Forzar numerico
    df_fase = df[cols_num + ['stunting12m']].copy()
    for c in df_fase.columns:
        df_fase[c] = pd.to_numeric(df_fase[c], errors='coerce')
    corr_f = df_fase[cols_num].corrwith(df_fase['stunting12m']).abs().dropna()
    if len(corr_f) > 0:
        fases_senal.append({
            'Fase':         fase.replace('F0_','').replace('F1_','').replace('F2_','')
                               .replace('F3_','').replace('F4_','').replace('F5_','')
                               .replace('F6_',''),
            'Max |r|':      round(corr_f.max(), 3),
            'Media |r|':    round(corr_f.mean(), 3),
            'N vars >0.05': int((corr_f > 0.05).sum()),
            'N vars total': len(corr_f),
        })

fs_df = pd.DataFrame(fases_senal)
print('Senal predictiva por fase (correlacion vs stunting a 12m):')
print(fs_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_fase = ['#8e44ad','#2980b9','#27ae60','#f39c12','#e67e22','#e74c3c','#95a5a6']
axes[0].barh(fs_df['Fase'], fs_df['Max |r|'],   color=colors_fase[:len(fs_df)], edgecolor='white')
for i, v in enumerate(fs_df['Max |r|']):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
axes[0].set_title('Maxima correlacion |r| por fase', fontweight='bold')
axes[0].set_xlabel('|r| maximo')
axes[0].set_xlim(0, fs_df['Max |r|'].max() * 1.2)

axes[1].barh(fs_df['Fase'], fs_df['Media |r|'], color=colors_fase[:len(fs_df)], edgecolor='white')
for i, v in enumerate(fs_df['Media |r|']):
    axes[1].text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)
axes[1].set_title('Correlacion media |r| por fase', fontweight='bold')
axes[1].set_xlabel('|r| promedio')
axes[1].set_xlim(0, fs_df['Media |r|'].max() * 1.2)

plt.suptitle('Que fase temporal aporta mas senal para predecir stunting a 12m?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mal_10_senal_por_fase.png', bbox_inches='tight')
plt.close()
print('Guardado: mal_10_senal_por_fase.png')

Senal predictiva por fase (correlacion vs stunting a 12m):
           Fase  Max |r|  Media |r|  N vars >0.05  N vars total
 Prenatal_Parto    0.161      0.037             7            41
     Nacimiento    0.270      0.116            23            34
Hospitalizacion    0.104      0.038            11            32
      40semanas    0.263      0.082            15            30
         3meses    0.381      0.136            15            26
         6meses    0.582      0.165            10            19
         9meses    0.641      0.207             9            15


Guardado: mal_10_senal_por_fase.png


## 7. Resumen Ejecutivo y Plan de Modelado

In [18]:
print('=' * 65)
print('RESUMEN EDA MALNUTRICION — PMCI Fundacion Canguro')
print('=' * 65)
print(f'''
DATASET
  Pacientes  : {len(df):,}
  Variables  : {df.shape[1]:,}
  Periodos   : 6 (aprox 1998-2023)

OUTCOMES (pacientes con dato en 12m)''')

for nombre, (bin_col, _, _) in OUTCOMES_BIN.items():
    n  = df[bin_col].notna().sum()
    p  = df[bin_col].mean() * 100
    print(f'  {nombre:<25}: {n:,} con dato | {p:.1f}% positivos')

pct_12m = (df['vino12m']==1).sum()/len(df)*100
print(f'''
DESERCION
  Llego a 12m : {(df['vino12m']==1).sum():,} ({pct_12m:.1f}%)
  Desercion   : ~{100-pct_12m:.1f}% - revisar si es MCAR o MAR

COBERTURA POR PERIODO
  Variables universales (>=50% en todos los periodos): {len(vars_universales)}
  Variables con cambio de protocolo (rango >30pp)   : {len(cambio)}
  Los periodos P1-P3 tienen cobertura REDUCIDA en seguimiento

PLAN DE MODELADO RECOMENDADO
  1. Modelo base universal : {len(vars_universales)} vars (robusto historicamente)
  2. Modelo extendido P4-P6: vars 2007-2023 (~47K pacientes)
  3. Cascada temporal F0->F1->F2->F3->F4->F5 con LightGBM
  4. Outcomes: stunting (principal), underweight, wasting
  5. Metricas: ROC-AUC + Sensibilidad + Especificidad
  6. Interpretar con SHAP values por fase
''')

# Exportar plan de features
import json
plan = {
    'vars_universales': vars_universales,
    'fases': {f: cols for f, cols in FASES_OK.items()},
    'outcomes': {
        'stunting':    'stunting12m',
        'underweight': 'underweight12m_b',
        'wasting':     'wasting12m',
    }
}
with open(OUT + 'feature_plan.json', 'w', encoding='utf-8') as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

print(f'Plan exportado: feature_plan.json')
print(f'Graficas generadas: mal_00 a mal_10 (.png)')

RESUMEN EDA MALNUTRICION — PMCI Fundacion Canguro

DATASET
  Pacientes  : 64,801
  Variables  : 758
  Periodos   : 6 (aprox 1998-2023)

OUTCOMES (pacientes con dato en 12m)
  Stunting (HAZ<-2)        : 30,953 con dato | 24.6% positivos
  Bajo peso (WAZ<-2)       : 29,897 con dato | 10.8% positivos
  Wasting (WHZ<-2)         : 29,828 con dato | 4.1% positivos

DESERCION
  Llego a 12m : 30,383 (46.9%)
  Desercion   : ~53.1% - revisar si es MCAR o MAR

COBERTURA POR PERIODO
  Variables universales (>=50% en todos los periodos): 156
  Variables con cambio de protocolo (rango >30pp)   : 467
  Los periodos P1-P3 tienen cobertura REDUCIDA en seguimiento

PLAN DE MODELADO RECOMENDADO
  1. Modelo base universal : 156 vars (robusto historicamente)
  2. Modelo extendido P4-P6: vars 2007-2023 (~47K pacientes)
  3. Cascada temporal F0->F1->F2->F3->F4->F5 con LightGBM
  4. Outcomes: stunting (principal), underweight, wasting
  5. Metricas: ROC-AUC + Sensibilidad + Especificidad
  6. Interpretar con 